# KernelForge: Training LLMs to Write Optimized CUDA Kernels

**A100 CUDA RL** — OpenEnv-compatible reinforcement learning environment for autonomous CUDA kernel optimization.

This notebook demonstrates two training paths:
1. **Path A: SFT on expert data** (works offline, no GPU backend needed)
2. **Path B: Stage 1 GRPO** (needs Modal/CoreWeave eval backend)

**Model**: `Qwen2.5-Coder-0.5B-Instruct` (default, fits any GPU). Set `KERNELFORGE_MODEL` for the full 30B MoE on A100 80GB.

**References**: [CUDA Agent](https://arxiv.org/abs/2602.24286) | [Dr. Kernel](https://arxiv.org/abs/2602.05885) | [KernelBench](https://arxiv.org/abs/2502.10517)

[![GitHub](https://img.shields.io/badge/GitHub-OCWC22%2FA100--CUDA--RL-blue)](https://github.com/OCWC22/A100-CUDA-RL)
[![HF Space](https://img.shields.io/badge/HF%20Space-AutomatedCUDA-yellow)](https://huggingface.co/spaces/AutomatedCUDA/A100-CUDA-RL)

In [ ]:
!pip install -q "trl>=0.7.0" "transformers>=4.40" "peft" "datasets" "accelerate>=1.4.0"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 1. Install Dependencies

In [ ]:
import os
os.chdir("/content")

!git clone https://github.com/OCWC22/A100-CUDA-RL.git 2>/dev/null || echo "Already cloned"
os.chdir("/content/A100-CUDA-RL")

import sys
sys.path.insert(0, "/content/A100-CUDA-RL")

print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')[:10]}...")

## 2. Clone Repo & Setup

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

# Environment configuration
os.environ["KERNELFORGE_TARGET_GPU"] = "A100"
os.environ["KERNELFORGE_TARGET_ARCH"] = "sm_80"

# Use small model by default (500M params, fits any GPU)
# For full 30B MoE, set: os.environ["KERNELFORGE_MODEL"] = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
os.environ["KERNELFORGE_DEV_MODEL"] = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

print("\nEnvironment configured for A100 target (sm_80)")

## 3. GPU Check & Environment Config

In [ ]:
import json
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

# --- Load model ---
MODEL_NAME = os.environ.get("KERNELFORGE_MODEL", "Qwen/Qwen2.5-Coder-0.5B-Instruct")
print(f"Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params:,} parameters")

# --- Load SFT dataset ---
sft_path = Path("datasets/doublegraph_sft.jsonl")
rows = []
with open(sft_path) as f:
    for line in f:
        row = json.loads(line.strip())
        if row.get("messages"):
            # Convert HF messages format to text
            text = ""
            for msg in row["messages"]:
                role = msg["role"]
                content = msg["content"]
                if role == "system":
                    text += f"<|system|>\n{content}\n"
                elif role == "user":
                    text += f"<|user|>\n{content}\n"
                elif role == "assistant":
                    text += f"<|assistant|>\n{content}\n"
            rows.append({"text": text})
        elif row.get("prompt") and row.get("completion"):
            rows.append({"text": f"<|user|>\n{row['prompt']}\n<|assistant|>\n{row['completion']}"})

dataset = Dataset.from_list(rows)
print(f"SFT dataset: {len(dataset)} examples")
print(f"Sample keys: {dataset.column_names}")
print(f"First example length: {len(rows[0]['text'])} chars")

## 4. Path A: SFT on doubleGraph Expert Data

Supervised fine-tuning on 192 expert CUDA kernel demonstrations from doubleGraph.
This works **offline** — no A100 eval backend needed. The model learns from expert-written kernels directly.

In [ ]:
# --- Run SFT Training ---
OUTPUT_DIR = "outputs/kernelforge-sft"

config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1,
    max_seq_length=2048,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    logging_steps=5,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=dataset,
)

print(f"Starting SFT training on {len(dataset)} examples...")
print(f"  Batch size: {config.per_device_train_batch_size} x {config.gradient_accumulation_steps} grad_accum")
print(f"  Learning rate: {config.learning_rate}")
print(f"  Max seq length: {config.max_seq_length}")

trainer.train()

# Save checkpoint
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nSFT complete! Checkpoint saved to {OUTPUT_DIR}")

In [ ]:
# --- Generate a CUDA kernel ---
prompt = """Write an optimized CUDA kernel for element-wise ReLU activation targeting NVIDIA A100 (sm_80).
The kernel should maximize memory throughput using vectorized loads (float4) and handle edge cases for non-aligned sizes.

```cuda"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
        repetition_penalty=1.05,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=" * 60)
print("GENERATED CUDA KERNEL")
print("=" * 60)
print(generated)

## 5. Generate a CUDA Kernel (Inference)

Test the fine-tuned model by generating a CUDA kernel from a prompt.

In [ ]:
# --- OPTIONAL: Stage 1 GRPO (requires Modal backend) ---
# Uncomment and fill in your Modal credentials to run this section.

# import os
# os.environ["KERNELFORGE_EVAL_BACKEND"] = "modal"
# os.environ["MODAL_TOKEN_ID"] = "your-token-id"
# os.environ["MODAL_TOKEN_SECRET"] = "your-token-secret"
# os.environ["KERNELFORGE_STAGE"] = "stage1"
# os.environ["KERNELFORGE_STAGE1_MAX_STEPS"] = "5"      # Smoke test (production: 100)
# os.environ["KERNELFORGE_STAGE1_MAX_TURNS"] = "1"       # Single turn (production: 3)
# os.environ["CUDA_AGENT_STAGE1_SAMPLES"] = "4"          # 4 samples (production: 512)
# os.environ["KERNELFORGE_LOCAL_COMPILE"] = "1"           # Local nvcc pre-check
#
# from training.stage1_warmup import main as stage1_main
# stage1_main()

print("Stage 1 GRPO section skipped (uncomment above to run with Modal backend)")

## 6. (Optional) Path B: Stage 1 GRPO with Eval Backend

This path runs the full RL training loop with multi-turn rollouts. **Requires Modal credentials** for A100 kernel evaluation.

Skip this section if you don't have Modal set up.

## 7. Environment Overview

The KernelForge RL system architecture:

| Component | Description |
|-----------|-------------|
| **OpenEnv Environment** | `reset()` samples task, `step()` evaluates CUDA kernel |
| **Reward** | Discrete milestones: -1 (fail), 1 (correct), 2 (beats eager), 3 (beats torch.compile) |
| **Anti-hack** | 5 Dr. Kernel-inspired runtime checks prevent reward gaming |
| **Training** | 3-stage: warmup GRPO → RFT/SFT → curriculum GRPO with TRLOO correction |
| **Eval Backend** | Remote A100 via CoreWeave/Northflank (primary) or Modal (fallback) |
| **Search Hedge** | SkyDiscover/AdaEvolve evolutionary search (regex-only mutation) |

**Full documentation**: See `docs/KERNELFORGE_RL_ENVIRONMENT.md` in the repo.

**HuggingFace Space**: [AutomatedCUDA/A100-CUDA-RL](https://huggingface.co/spaces/AutomatedCUDA/A100-CUDA-RL)